# 00_Load data and build the modeling table
Converts the October 2019 event log to Parquet, profiles data quality, defines the cart-to-purchase label, builds session-level features, and saves `model_table.parquet` for the modeling notebook.

## 1. Load data

Converts the raw `2019-Oct.csv.gz` to Parquet (column names and types inferred automatically). Parquet is columnar, so the queries below run much faster than on the compressed CSV. The conversion is skipped automatically if the Parquet file already exists.

In [1]:
import os
import duckdb

duckdb.sql("SET enable_progress_bar = false")
raw_path = '../data/raw/2019-Oct.csv.gz'
parquet_path = '../data/processed/events_2019_10.parquet'

# Skip if the Parquet file already exists; convert only on the first run
if os.path.exists(parquet_path):
    print('Parquet already exists, skipping conversion.')
else:
    duckdb.sql(f"""
    COPY (SELECT * FROM read_csv_auto('{raw_path}'))
    TO '{parquet_path}' (FORMAT PARQUET)
    """)
    print('Conversion done.')

Parquet already exists, skipping conversion.


### Register the `events` view
Creates a view named `events` over the Parquet file so every query can use `FROM events`. A view stores only the query, not the data; the file is read when a query runs. Re-run this cell after a kernel restart.

In [2]:
duckdb.sql("""
CREATE OR REPLACE VIEW events AS
SELECT * FROM '../data/processed/events_2019_10.parquet'
""")

## 2. Data profiling

### Schema check
`DESCRIBE` lists each column and its type. `null = YES` means NULLs are allowed, not that any exist.

In [3]:
duckdb.sql("DESCRIBE SELECT * FROM events").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_time    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

### Preview
First 10 rows in time order.

In [4]:
duckdb.sql("SELECT * FROM events ORDER BY event_time, user_session, product_id LIMIT 10").show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────────────────────┬──────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │            category_code            │  brand   │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │   int64    │        int64        │               varchar               │ varchar  │ double  │   int64   │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────────────────────┼──────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-10-01 00:00:00 │ view       │   44600062 │ 2103807459595387724 │ NULL                                │ shiseido │   35.79 │ 541312140 │ 72d76fde-8bb3-4e00-8c23-a032dfed738c │
│ 2019-10-01 00:00:00 │ view       │    3900821 │ 2053013552326770905 │ appliances.environ

### Dataset size
Total events, distinct users and sessions, and the time range covered.

In [5]:
duckdb.sql("""
SELECT COUNT(*) AS n_events,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT user_session) AS n_sessions,
       MIN(event_time), MAX(event_time)
FROM events
""").show()

┌──────────┬─────────┬────────────┬─────────────────────┬─────────────────────┐
│ n_events │ n_users │ n_sessions │   min(event_time)   │   max(event_time)   │
│  int64   │  int64  │   int64    │      timestamp      │      timestamp      │
├──────────┼─────────┼────────────┼─────────────────────┼─────────────────────┤
│ 42448764 │ 3022290 │    9244421 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │
└──────────┴─────────┴────────────┴─────────────────────┴─────────────────────┘



### Column-level summary
`SUMMARIZE` reports min, max, approximate distinct count, mean, std, quartiles and null percentage for every column. `approx_unique` is an estimate; exact distinct counts are in the cell above.

In [6]:
duckdb.sql("SUMMARIZE SELECT * FROM events").show()

┌───────────────┬─────────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────────┬────────────────────────────┬────────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬──────────┬─────────────────┐
│  column_name  │ column_type │                 min                  │                 max                  │ approx_unique │            avg             │          std           │            q25             │            q50             │            q75             │  count   │ null_percentage │
│    varchar    │   varchar   │               varchar                │               varchar                │     int64     │          varchar           │        varchar         │          varchar           │          varchar           │          varchar           │  int64   │  decimal(9,2)   │
├───────────────┼─────────────┼──────────────────────────────────────┼──────────────────────────────────────┼───

### Event type distribution
Count and share of each `event_type`. `SUM(COUNT(*)) OVER ()` gives the grand total used as the denominator.

In [7]:
duckdb.sql("""
SELECT event_type, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM events
GROUP BY event_type
ORDER BY n DESC
""").show()

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘



### Events per session
Aggregates to one row per session, then reports single-event sessions and the mean, median and max events per session.

In [8]:
duckdb.sql("""
WITH s AS (
    SELECT user_session, COUNT(*) AS n_events 
    FROM events
    GROUP BY user_session
)
SELECT COUNT(*) AS n_sessions,
       SUM(CASE WHEN n_events = 1 THEN 1 ELSE 0 END) AS n_single_event_sessions,  -- sessions with only 1 event
       ROUND(AVG(n_events), 2) AS avg_events,
       MEDIAN(n_events) AS median_events,
       MAX(n_events) AS max_events
FROM s
""").show()

┌────────────┬─────────────────────────┬────────────┬───────────────┬────────────┐
│ n_sessions │ n_single_event_sessions │ avg_events │ median_events │ max_events │
│   int64    │         int128          │   double   │    double     │   int64    │
├────────────┼─────────────────────────┼────────────┼───────────────┼────────────┤
│    9244422 │                 3269577 │       4.59 │           2.0 │       1159 │
└────────────┴─────────────────────────┴────────────┴───────────────┴────────────┘



### NULL count for every column
`COUNT(col)` skips NULLs, so `COUNT(*) - COUNT(col)` is the number of NULLs in that column.

In [9]:
duckdb.sql("""
SELECT COUNT(*) - COUNT(event_time)    AS null_event_time,
       COUNT(*) - COUNT(event_type)    AS null_event_type,
       COUNT(*) - COUNT(product_id)    AS null_product_id,
       COUNT(*) - COUNT(category_id)   AS null_category_id,
       COUNT(*) - COUNT(category_code) AS null_category_code,
       COUNT(*) - COUNT(brand)         AS null_brand,
       COUNT(*) - COUNT(price)         AS null_price,
       COUNT(*) - COUNT(user_id)       AS null_user_id,
       COUNT(*) - COUNT(user_session)  AS null_user_session
FROM events
""").show()

┌─────────────────┬─────────────────┬─────────────────┬──────────────────┬────────────────────┬────────────┬────────────┬──────────────┬───────────────────┐
│ null_event_time │ null_event_type │ null_product_id │ null_category_id │ null_category_code │ null_brand │ null_price │ null_user_id │ null_user_session │
│      int64      │      int64      │      int64      │      int64       │       int64        │   int64    │   int64    │    int64     │       int64       │
├─────────────────┼─────────────────┼─────────────────┼──────────────────┼────────────────────┼────────────┼────────────┼──────────────┼───────────────────┤
│               0 │               0 │               0 │                0 │           13515609 │    6113008 │          0 │            0 │                 2 │
└─────────────────┴─────────────────┴─────────────────┴──────────────────┴────────────────────┴────────────┴────────────┴──────────────┴───────────────────┘



### Missing rate for key columns
Share of NULLs in `category_code`, `brand` and `user_session`, averaging a 0/1 flag gives the share.

In [10]:
duckdb.sql("""
SELECT ROUND(100.0 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_category,
       ROUND(100.0 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_brand,
       ROUND(100.0 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 4) AS pct_missing_session
FROM events
""").show()

┌──────────────────────┬───────────────────┬─────────────────────┐
│ pct_missing_category │ pct_missing_brand │ pct_missing_session │
│        double        │      double       │       double        │
├──────────────────────┼───────────────────┼─────────────────────┤
│                31.84 │              14.4 │                 0.0 │
└──────────────────────┴───────────────────┴─────────────────────┘



**Key findings: data profiling**
- 42.4M events covering all of October 2019.
- 3.02M distinct users and 9.24M distinct sessions (~3.1 sessions per user).
- Views dominate (96.1% of events), cart and purchase are 2.2% and 1.7%. 
- Purchases (743K) are close to cart adds (927K), so many purchases may lack a preceding cart event; checked at the session level below.
- Missing values: `category_code` 13.5M (31.8%), `brand` 6.1M (14.4%). All other columns are complete except `user_session` (2 rows). `GROUP BY user_session` treats these as one extra group, hence 9,244,422 sessions vs. 9,244,421 distinct.
- `price` has a minimum of 0.0 (to check before any revenue metric) and is right-skewed (mean 290 vs. median 163).
- 35.4% of sessions contain a single event. Events per session: median 2, mean 4.59, max 1,159.

## 3. Session-level cart and purchase behavior

### Cart × purchase cross-tab and duplicate rows
Flags each session with `has_cart` / `has_purchase` and counts the four combinations, then counts fully duplicated rows (total rows minus distinct rows).

In [11]:
duckdb.sql("""
WITH s AS (
-- flatten the event level into the session level, with one row per session
    SELECT user_session,
           MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
           MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM events
    GROUP BY user_session
)

-- Count the four combinations of (has_cart, has_purchase) for the session table
-- 0,0: browsed only / 0,1: purchased without cart / 1,0: added to cart without purchase / 1,1: added to cart and purchased
SELECT has_cart, has_purchase, COUNT(*) AS n_sessions,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM s
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

# Count of duplicate rows
duckdb.sql("""
SELECT (SELECT COUNT(*) FROM events)
     - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM events)) AS n_duplicate_rows
""").show()

┌──────────┬──────────────┬────────────┬────────┐
│ has_cart │ has_purchase │ n_sessions │  pct   │
│  int32   │    int32     │   int64    │ double │
├──────────┼──────────────┼────────────┼────────┤
│        0 │            0 │    8333625 │  90.15 │
│        0 │            1 │     337699 │   3.65 │
│        1 │            0 │     281237 │   3.04 │
│        1 │            1 │     291861 │   3.16 │
└──────────┴──────────────┴────────────┴────────┘

┌──────────────────┐
│ n_duplicate_rows │
│      int64       │
├──────────────────┤
│            30220 │
└──────────────────┘



### Duplicates by event type
Duplicates per `event_type`, defined as rows identical on time, type, product, user, session and price.

In [12]:
duckdb.sql("""
WITH d AS (
    SELECT event_type,
           COUNT(*) AS n_rows,
           COUNT(*) - COUNT(DISTINCT (event_time, event_type, product_id, user_id, user_session, price)) AS n_duplicates
    FROM events
    GROUP BY event_type
)
SELECT event_type,
       n_rows,
       n_duplicates,
       ROUND(100.0 * n_duplicates / n_rows, 3) AS pct_duplicates
FROM d
ORDER BY pct_duplicates DESC
""").show()

┌────────────┬──────────┬──────────────┬────────────────┐
│ event_type │  n_rows  │ n_duplicates │ pct_duplicates │
│  varchar   │  int64   │    int64     │     double     │
├────────────┼──────────┼──────────────┼────────────────┤
│ cart       │   926516 │        28073 │           3.03 │
│ purchase   │   742849 │           76 │           0.01 │
│ view       │ 40779399 │         2071 │          0.005 │
└────────────┴──────────┴──────────────┴────────────────┘



**Key findings: sessions and duplicates**
- 90.2% of sessions are browse-only (no cart, no purchase).
- 30,220 exact duplicate rows (0.07% of events): 92.9% are cart events (3.0% of cart rows) and only 76 are purchases. With no quantity column, same-second repeat carts may be multiple units rather than logging errors. Session-level flags are unaffected either way.

### Build the `sessions` table
Saves the session-level flags as a table so later queries don't rescan the raw events. Expected rows: 9,244,422 (includes the NULL-session group).

In [13]:
# Flatten the event level into the session level and save it as a table named sessions
duckdb.sql("""
CREATE OR REPLACE TABLE sessions AS
SELECT user_session,
       MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
       MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
FROM events
GROUP BY user_session
""")

# View the first 10 rows
duckdb.sql("SELECT * FROM sessions ORDER BY user_session LIMIT 10").show()

# Confirm the row count, which should be 9,244,422
duckdb.sql("SELECT COUNT(*) AS n_rows FROM sessions").show()

┌──────────────────────────────────────┬──────────┬──────────────┐
│             user_session             │ has_cart │ has_purchase │
│               varchar                │  int32   │    int32     │
├──────────────────────────────────────┼──────────┼──────────────┤
│ 00000042-3e3f-42f9-810d-f3d264139c50 │        0 │            0 │
│ 00000056-a206-40dd-b174-a072550fa38c │        0 │            0 │
│ 00000083-8816-4d58-a9b8-f52f54186edc │        1 │            1 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │        1 │            1 │
│ 000003eb-b63e-45d9-9f26-f229057c654a │        0 │            0 │
│ 0000047e-bdcc-4854-9e8d-9da7f84010ae │        0 │            0 │
│ 00000809-9101-4e4b-9795-e6cbafccfe19 │        0 │            0 │
│ 00000a05-fa4e-4486-b5e8-c8bffa63b422 │        0 │            0 │
│ 00000aaa-d774-49bc-9c31-0c9f6e1c2f0a │        0 │            0 │
│ 00000e03-279f-4a23-acb4-40d06b576299 │        0 │            0 │
└──────────────────────────────────────┴──────────┴───────────

### Cart × purchase with conditional percentages
`pct_of_all` is each combination's share of all sessions; `pct_within_cart_group` is its share among sessions with the same `has_cart` value, which gives the cart conversion and abandonment rates.

In [14]:
# Number of sessions for each of the four combinations
duckdb.sql("""
SELECT has_cart,
       has_purchase,
       COUNT(*) AS n_sessions,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_all,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY has_cart), 2) AS pct_within_cart_group
FROM sessions
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

┌──────────┬──────────────┬────────────┬────────────┬───────────────────────┐
│ has_cart │ has_purchase │ n_sessions │ pct_of_all │ pct_within_cart_group │
│  int32   │    int32     │   int64    │   double   │        double         │
├──────────┼──────────────┼────────────┼────────────┼───────────────────────┤
│        0 │            0 │    8333625 │      90.15 │                 96.11 │
│        0 │            1 │     337699 │       3.65 │                  3.89 │
│        1 │            0 │     281237 │       3.04 │                 49.07 │
│        1 │            1 │     291861 │       3.16 │                 50.93 │
└──────────┴──────────────┴────────────┴────────────┴───────────────────────┘



**Key findings: cart conversion**
- Sessions without a cart: 96.11% made no purchase; 3.89% purchased without a cart event.
- Sessions with a cart (573,098): 50.93% purchased (**cart conversion rate**), 49.07% did not (**cart abandonment rate**).
- Of 629,560 purchasing sessions, 53.6% have no cart event in the same session; the cart step may happen in an earlier session or be skipped (e.g. "buy now").
- **Scope:** this project predicts whether a cart session ends in a purchase. Purchases without a cart event are out of scope.

## 4. Label: purchase after the first cart
For each session with a cart event, `label = 1` if a purchase occurs at or after the first cart event, otherwise `0`. This needs two inputs: the first cart time per session, and whether a purchase follows it.

### First cart time per session
Earliest cart event and the user for each session with a cart; the reference point for the label and all features.

In [15]:
duckdb.sql("""
CREATE OR REPLACE TABLE first_cart AS
SELECT user_session,
       ANY_VALUE(user_id) AS user_id,        -- The same session belongs to the same user so take any value
       MIN(event_time) AS first_cart_time    -- earliest cart time
FROM events
WHERE event_type = 'cart'                    -- Only look at cart events
GROUP BY user_session                        -- One row per session
""")

duckdb.sql("SELECT * FROM first_cart ORDER BY user_session LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) AS n FROM first_cart").show()

┌──────────────────────────────────────┬───────────┬─────────────────────┐
│             user_session             │  user_id  │   first_cart_time   │
│               varchar                │   int64   │      timestamp      │
├──────────────────────────────────────┼───────────┼─────────────────────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 546521725 │ 2019-10-06 11:33:54 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │ 560486342 │ 2019-10-25 08:34:23 │
│ 00005026-a9d1-4e2b-8290-3cc14e4bad89 │ 520649833 │ 2019-10-15 08:03:21 │
│ 00005731-b1c2-4291-a2e4-3b3c87d85114 │ 531956094 │ 2019-10-14 13:23:19 │
│ 000081ea-9376-4eb8-a69b-0b592765a21d │ 513622224 │ 2019-10-24 09:06:14 │
└──────────────────────────────────────┴───────────┴─────────────────────┘

┌────────┐
│   n    │
│ int64  │
├────────┤
│ 573098 │
└────────┘



### Spot check: one session's timeline
Lists every event of one cart-and-purchase session against its first cart time, to confirm the join and `after_first_cart` logic on a typical case. Edge cases (multiple carts, purchase before the first cart) are not covered here.

In [16]:
# Pick a session that has both a cart and a purchase, and look at the relationship between each event and the time of the first cart addition
duckdb.sql("""
SELECT f.user_session,
       e.event_time,
       e.event_type,
       e.product_id,
       f.first_cart_time,
       e.event_time >= f.first_cart_time AS after_first_cart   -- Whether this event occurred after the first cart addition
FROM first_cart f
JOIN events e
  ON e.user_session = f.user_session          -- Join the two tables using session
WHERE f.user_session = (
    SELECT user_session FROM sessions
    WHERE has_cart = 1 AND has_purchase = 1
    ORDER BY user_session                     -- Lock in the same session so the result is consistent every time
    LIMIT 1
)
ORDER BY e.event_time, e.event_type, e.product_id
""").show()

┌──────────────────────────────────────┬─────────────────────┬────────────┬────────────┬─────────────────────┬──────────────────┐
│             user_session             │     event_time      │ event_type │ product_id │   first_cart_time   │ after_first_cart │
│               varchar                │      timestamp      │  varchar   │   int64    │      timestamp      │     boolean      │
├──────────────────────────────────────┼─────────────────────┼────────────┼────────────┼─────────────────────┼──────────────────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 2019-10-06 11:24:45 │ view       │    1004768 │ 2019-10-06 11:33:54 │ false            │
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 2019-10-06 11:25:54 │ view       │    1005098 │ 2019-10-06 11:33:54 │ false            │
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 2019-10-06 11:25:59 │ view       │    1005073 │ 2019-10-06 11:33:54 │ false            │
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 2019-10-06 11:26:39 │ view       │    1004871 │ 2

### Build the label table `cart_sessions`
Applies the same join to all cart sessions, one row per session.

In [17]:
# Build one row per session that added something to the cart.
# label = 1 if the session has a purchase at or after its first cart event, else 0.
duckdb.sql("""
CREATE OR REPLACE TABLE cart_sessions AS
SELECT f.user_session,
       f.user_id,
       f.first_cart_time,
       MAX(CASE WHEN e.event_type = 'purchase'
                 AND e.event_time >= f.first_cart_time
                THEN 1 ELSE 0 END) AS label
FROM first_cart f
JOIN events e
  ON e.user_session = f.user_session
GROUP BY f.user_session, f.user_id, f.first_cart_time
""")

### Check the labels
Preview and label distribution.

In [18]:
duckdb.sql("SELECT * FROM cart_sessions ORDER BY user_session LIMIT 5").show()

duckdb.sql("""
SELECT label, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM cart_sessions
GROUP BY label
ORDER BY label
""").show()

┌──────────────────────────────────────┬───────────┬─────────────────────┬───────┐
│             user_session             │  user_id  │   first_cart_time   │ label │
│               varchar                │   int64   │      timestamp      │ int32 │
├──────────────────────────────────────┼───────────┼─────────────────────┼───────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │ 546521725 │ 2019-10-06 11:33:54 │     1 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │ 560486342 │ 2019-10-25 08:34:23 │     1 │
│ 00005026-a9d1-4e2b-8290-3cc14e4bad89 │ 520649833 │ 2019-10-15 08:03:21 │     1 │
│ 00005731-b1c2-4291-a2e4-3b3c87d85114 │ 531956094 │ 2019-10-14 13:23:19 │     0 │
│ 000081ea-9376-4eb8-a69b-0b592765a21d │ 513622224 │ 2019-10-24 09:06:14 │     1 │
└──────────────────────────────────────┴───────────┴─────────────────────┴───────┘

┌───────┬────────┬────────┐
│ label │   n    │  pct   │
│ int32 │ int64  │ double │
├───────┼────────┼────────┤
│     0 │ 282868 │  49.36 │
│     1 │ 290229 │  50.64 │
└

**Label check**
- 573,097 cart sessions, one fewer than `first_cart` (573,098), most likely the NULL `user_session` group, which never matches in a join.
- Classes are balanced: 50.6% label 1, 49.4% label 0.
- 290,229 positives vs. 291,861 sessions with both cart and purchase: 1,632 sessions purchased only before their first cart and are labeled 0.

## 5. Features

### Feature set 1 `feat_pre_cart`: behavior before the first cart
Uses only events before the first cart (no leakage): number of views, distinct products viewed, seconds from session start to first cart, and hour and day of week of the cart.

In [19]:
# Behavior of each cart session prior to the first cart addition
duckdb.sql("""
CREATE OR REPLACE TABLE feat_pre_cart AS
SELECT f.user_session,
       -- Number of views before the cart
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'view') AS n_views_before_cart,
       -- Number of distinct products viewed before the cart
       COUNT(DISTINCT e.product_id) FILTER (WHERE e.event_type = 'view') AS n_products_viewed_before_cart,
       -- Seconds elapsed from the session start to the first cart
       COALESCE(DATE_DIFF('second', MIN(e.event_time), f.first_cart_time), 0) AS secs_to_first_cart,
       -- Time features
       HOUR(f.first_cart_time) AS cart_hour,
       DAYOFWEEK(f.first_cart_time) AS cart_dow
FROM first_cart f
LEFT JOIN events e
  ON e.user_session = f.user_session
 AND e.event_time < f.first_cart_time        -- join only events prior to the cart
GROUP BY f.user_session, f.first_cart_time
""")

duckdb.sql("SELECT * FROM feat_pre_cart ORDER BY user_session LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_pre_cart").show()

┌──────────────────────────────────────┬─────────────────────┬───────────────────────────────┬────────────────────┬───────────┬──────────┐
│             user_session             │ n_views_before_cart │ n_products_viewed_before_cart │ secs_to_first_cart │ cart_hour │ cart_dow │
│               varchar                │        int64        │             int64             │       int64        │   int64   │  int64   │
├──────────────────────────────────────┼─────────────────────┼───────────────────────────────┼────────────────────┼───────────┼──────────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │                  16 │                            14 │                549 │        11 │        0 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │                   4 │                             3 │                221 │         8 │        5 │
│ 00005026-a9d1-4e2b-8290-3cc14e4bad89 │                   2 │                             2 │                 62 │         8 │        2 │
│ 00005731-b1c2-4291-a2e4-3

### Feature set 2 `feat_cart_product`: the first carted product
Product ID, price, category ID and a missing-brand flag for the product at the first cart event. If several products were carted in the same second, `ANY_VALUE` picks one arbitrarily.

In [20]:
# Attributes of the product added to the first cart
duckdb.sql("""
CREATE OR REPLACE TABLE feat_cart_product AS
SELECT f.user_session,
       ANY_VALUE(e.product_id) AS cart_product_id,
       ANY_VALUE(e.price) AS cart_price,
       ANY_VALUE(e.category_id) AS cart_category_id,
       ANY_VALUE(CASE WHEN e.brand IS NULL THEN 1 ELSE 0 END) AS cart_brand_missing
FROM first_cart f
JOIN events e
  ON e.user_session = f.user_session
 AND e.event_type = 'cart'
 AND e.event_time = f.first_cart_time     -- Join only the event corresponding to the first cart addition
GROUP BY f.user_session
""")

duckdb.sql("SELECT * FROM feat_cart_product ORDER BY user_session LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_cart_product").show()

┌──────────────────────────────────────┬─────────────────┬────────────┬─────────────────────┬────────────────────┐
│             user_session             │ cart_product_id │ cart_price │  cart_category_id   │ cart_brand_missing │
│               varchar                │      int64      │   double   │        int64        │       int32        │
├──────────────────────────────────────┼─────────────────┼────────────┼─────────────────────┼────────────────────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │         1004870 │     289.52 │ 2053013555631882655 │                  0 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │         1004834 │      171.9 │ 2053013555631882655 │                  0 │
│ 00005026-a9d1-4e2b-8290-3cc14e4bad89 │         3600163 │     182.41 │ 2053013563810775923 │                  0 │
│ 00005731-b1c2-4291-a2e4-3b3c87d85114 │        10700055 │      38.35 │ 2053013561277416167 │                  1 │
│ 000081ea-9376-4eb8-a69b-0b592765a21d │         1004856 │     131.51 │ 20530135

## 6. Model table

### Assemble `model_table`
Joins the labels and both feature sets on `user_session` and adds a time-based split: first carts before 2019-10-25 go to train, later ones to test. Then checks row counts and the label rate per split.

In [21]:
# Join the label and the two batches of features into a single table by session
duckdb.sql("""
CREATE OR REPLACE TABLE model_table AS
SELECT c.user_session,
       c.user_id,
       c.first_cart_time,
       c.label,
       -- Behavior before the cart
       p.n_views_before_cart,
       p.n_products_viewed_before_cart,
       p.secs_to_first_cart,
       p.cart_hour,
       p.cart_dow,
       -- Product added to the cart
       cp.cart_product_id,
       cp.cart_price,
       cp.cart_category_id,
       cp.cart_brand_missing,
       -- Time-based split: Before 10/25 as train, after as test
       CASE WHEN c.first_cart_time < TIMESTAMP '2019-10-25' THEN 'train' ELSE 'test' END AS split
FROM cart_sessions c
JOIN feat_pre_cart p      ON p.user_session = c.user_session
JOIN feat_cart_product cp ON cp.user_session = c.user_session
""")

duckdb.sql("SELECT COUNT(*) AS n FROM model_table").show()

# Number of rows and purchase rate for each split
duckdb.sql("""
SELECT split,
       COUNT(*) AS n,
       ROUND(100.0 * AVG(label), 2) AS pct_label_1
FROM model_table
GROUP BY split
ORDER BY split DESC      -- Train before, test after
""").show()

┌────────┐
│   n    │
│ int64  │
├────────┤
│ 573097 │
└────────┘

┌─────────┬────────┬─────────────┐
│  split  │   n    │ pct_label_1 │
│ varchar │ int64  │   double    │
├─────────┼────────┼─────────────┤
│ train   │ 463222 │       50.32 │
│ test    │ 109875 │       51.98 │
└─────────┴────────┴─────────────┘



### Save `model_table`
Writes the table to Parquet for the modeling notebook.

In [22]:
duckdb.sql("""
COPY model_table TO '../data/processed/model_table.parquet' (FORMAT PARQUET)
""")

### Feature set 3 `feat_user_history`: prior user behavior
For each cart session, counts the same user's purchases, carts and sessions in *other* sessions before the first cart time (no leakage). Reads from the saved `model_table` and is saved as a separate Parquet file.

In [23]:
# User behavior in other sessions prior to this first cart addition
duckdb.sql("""
CREATE OR REPLACE TABLE feat_user_history AS
SELECT m.user_session,
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'purchase') AS prior_purchases,
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'cart')     AS prior_carts,
       COUNT(DISTINCT e.user_session)                               AS prior_sessions
FROM '../data/processed/model_table.parquet' m
LEFT JOIN events e
  ON e.user_id = m.user_id                    -- Same user
 AND e.event_time < m.first_cart_time         -- Only look before this cart to prevent data leakage
 AND e.user_session <> m.user_session         -- Exclude the current session (already captured in pre-cart features)
GROUP BY m.user_session
""")

duckdb.sql("SELECT * FROM feat_user_history ORDER BY user_session LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_user_history").show()

duckdb.sql("""
COPY feat_user_history TO '../data/processed/feat_user_history.parquet' (FORMAT PARQUET)
""")

┌──────────────────────────────────────┬─────────────────┬─────────────┬────────────────┐
│             user_session             │ prior_purchases │ prior_carts │ prior_sessions │
│               varchar                │      int64      │    int64    │     int64      │
├──────────────────────────────────────┼─────────────────┼─────────────┼────────────────┤
│ 00000083-8816-4d58-a9b8-f52f54186edc │               0 │           0 │              0 │
│ 000001fd-1f89-45e8-a3ce-fe3218cabfad │               4 │           6 │             10 │
│ 00005026-a9d1-4e2b-8290-3cc14e4bad89 │               0 │           0 │              2 │
│ 00005731-b1c2-4291-a2e4-3b3c87d85114 │               1 │           1 │              5 │
│ 000081ea-9376-4eb8-a69b-0b592765a21d │               0 │           0 │              0 │
└──────────────────────────────────────┴─────────────────┴─────────────┴────────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       573097 │
└─────────────